In [1]:
import os

structure = {
    "openqkd": {
        "__init__.py": "",
        "core": {
            "__init__.py": "",
            "qkd_param.py": "",
            "utils.py": "",
        },
        "solvers": {
            "__init__.py": "",
            "fw2step.py": "",
            "math_solver.py": "",
        },
        "modules": {
            "__init__.py": "",
            "description": {
                "__init__.py": "",
                "bb84_description.py": "",
            },
            "channel": {
                "__init__.py": "",
                "bb84_channel.py": "",
            },
            "keyrate": {
                "__init__.py": "",
                "bb84_keyrate.py": "",
            },
        },
        "presets": {
            "__init__.py": "",
            "bb84_preset.py": "",
        },
        "optimizer": {
            "__init__.py": "",
            "main_iteration.py": "",
        },
    },
    "tests": {
        "__init__.py": "",
        "test_utils.py": "",
        "test_fw2step.py": "",
        "test_bb84.py": "",
    },
    "notebooks": {
        "01_bb84_example.ipynb": "",
    },
    "requirements.txt": "numpy\nscipy\ntoqito\ncvxpy\nclarabel\nscs\nmosek\njupyter\nmatplotlib\nipywidgets\nipykernel\n",
    "README.md": "# openQKD Python\nPort do framework openQKDsecurity (MATLAB) para Python.\n",
}

def create_structure(base_path, tree):
    for name, content in tree.items():
        path = os.path.join(base_path, name)
        if isinstance(content, dict):
            os.makedirs(path, exist_ok=True)
            create_structure(path, content)
        else:
            with open(path, "w") as f:
                f.write(content)
            print(f"  criado: {path}")

base = "openqkd_python"
os.makedirs(base, exist_ok=True)
create_structure(base, structure)
print("\n✅ Estrutura criada com sucesso!")


  criado: openqkd_python\openqkd\__init__.py
  criado: openqkd_python\openqkd\core\__init__.py
  criado: openqkd_python\openqkd\core\qkd_param.py
  criado: openqkd_python\openqkd\core\utils.py
  criado: openqkd_python\openqkd\solvers\__init__.py
  criado: openqkd_python\openqkd\solvers\fw2step.py
  criado: openqkd_python\openqkd\solvers\math_solver.py
  criado: openqkd_python\openqkd\modules\__init__.py
  criado: openqkd_python\openqkd\modules\description\__init__.py
  criado: openqkd_python\openqkd\modules\description\bb84_description.py
  criado: openqkd_python\openqkd\modules\channel\__init__.py
  criado: openqkd_python\openqkd\modules\channel\bb84_channel.py
  criado: openqkd_python\openqkd\modules\keyrate\__init__.py
  criado: openqkd_python\openqkd\modules\keyrate\bb84_keyrate.py
  criado: openqkd_python\openqkd\presets\__init__.py
  criado: openqkd_python\openqkd\presets\bb84_preset.py
  criado: openqkd_python\openqkd\optimizer\__init__.py
  criado: openqkd_python\openqkd\optimi

In [2]:
for root, dirs, files in os.walk(base):
    level = root.replace(base, "").count(os.sep)
    print("  " * level + os.path.basename(root) + "/")
    for f in files:
        print("  " * (level + 1) + f)


openqkd_python/
  README.md
  requirements.txt
  notebooks/
    01_bb84_example.ipynb
  openqkd/
    __init__.py
    core/
      qkd_param.py
      utils.py
      __init__.py
    modules/
      __init__.py
      channel/
        bb84_channel.py
        __init__.py
      description/
        bb84_description.py
        __init__.py
      keyrate/
        bb84_keyrate.py
        __init__.py
    optimizer/
      main_iteration.py
      __init__.py
    presets/
      bb84_preset.py
      __init__.py
    solvers/
      fw2step.py
      math_solver.py
      __init__.py
  tests/
    test_bb84.py
    test_fw2step.py
    test_utils.py
    __init__.py


In [3]:
import sys, numpy, scipy, cvxpy
from importlib.metadata import version

print(f"Python  : {sys.version.split()[0]}")
print(f"NumPy   : {numpy.__version__}")
print(f"SciPy   : {scipy.__version__}")
print(f"CVXPY   : {cvxpy.__version__}")
print(f"toqito  : {version('toqito')}")
print(f"Solvers : {cvxpy.installed_solvers()}")


Python  : 3.11.9
NumPy   : 2.2.6
SciPy   : 1.15.3
CVXPY   : 1.7.1
toqito  : 1.1.3
Solvers : ['CLARABEL', 'CVXOPT', 'GLPK', 'GLPK_MI', 'MOSEK', 'OSQP', 'SCIPY', 'SCS']


In [4]:
import numpy as np
import scipy
import cvxpy as cp
from scipy.linalg import logm
from importlib.metadata import version

# 1. Verifica numpy/scipy
rho = np.array([[0.5, 0.5], [0.5, 0.5]])
print(f"✓ NumPy {np.__version__}, SciPy {scipy.__version__}")
print(f"  logm(rho) OK: shape={logm(rho + 1e-9*np.eye(2)).shape}")

# 2. Verifica cvxpy + CLARABEL
X = cp.Variable((2,2), hermitian=True)
prob = cp.Problem(cp.Minimize(cp.real(cp.trace(X))), [X >> np.eye(2)])
prob.solve(solver=cp.CLARABEL)
print(f"✓ CVXPY + CLARABEL OK: valor={prob.value:.4f} (esperado ~2.0)")

# 3. Tenta MOSEK mas não quebra se não tiver licença
try:
    prob.solve(solver=cp.MOSEK)
    print(f"✓ CVXPY + MOSEK OK: valor={prob.value:.4f}")
except Exception as e:
    print(f"⚠ MOSEK sem licença ainda (esperado): {type(e).__name__}")

# 4. Verifica toqito
from toqito.channels import partial_trace
rho_AB = np.kron(rho, rho)
rho_A  = partial_trace(rho_AB, [1], [2, 2])
print(f"✓ toqito partial_trace OK: shape={rho_A.shape}")

# 5. Verifica toqito version
print(f"✓ toqito versão: {version('toqito')}")

print("\n✅ Ambiente pronto para o port do openQKDsecurity!")


✓ NumPy 2.2.6, SciPy 1.15.3
  logm(rho) OK: shape=(2, 2)
✓ CVXPY + CLARABEL OK: valor=2.0000 (esperado ~2.0)
⚠ MOSEK sem licença ainda (esperado): Error
✓ toqito partial_trace OK: shape=(2, 2)
✓ toqito versão: 1.1.3

✅ Ambiente pronto para o port do openQKDsecurity!


In [5]:
import os

structure = {
    "openqkd": {
        "__init__.py": "",
        "core": {
            "__init__.py": "",
            "qkd_param.py": "",
            "utils.py": "",
        },
        "solvers": {
            "__init__.py": "",
            "fw2step.py": "",
            "math_solver.py": "",
        },
        "modules": {
            "__init__.py": "",
            "description": {
                "__init__.py": "",
                "bb84_description.py": "",
            },
            "channel": {
                "__init__.py": "",
                "bb84_channel.py": "",
            },
            "keyrate": {
                "__init__.py": "",
                "bb84_keyrate.py": "",
            },
        },
        "presets": {
            "__init__.py": "",
            "bb84_preset.py": "",
        },
        "optimizer": {
            "__init__.py": "",
            "main_iteration.py": "",
        },
    },
    "tests": {
        "__init__.py": "",
        "test_utils.py": "",
        "test_fw2step.py": "",
        "test_bb84.py": "",
    },
}

def create_structure(base_path, tree):
    for name, content in tree.items():
        path = os.path.join(base_path, name)
        if isinstance(content, dict):
            os.makedirs(path, exist_ok=True)
            create_structure(path, content)
        else:
            # exist_ok: não sobrescreve arquivos que já têm conteúdo
            if not os.path.exists(path):
                with open(path, "w") as f:
                    f.write(content)
                print(f"  criado: {path}")
            else:
                print(f"  já existe: {path}")

base = r"C:\Users\Mario\Documents\OpenQKD\openqkd_python"
create_structure(base, structure)
print("\n✅ Estrutura verificada!")


  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\__init__.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\core\__init__.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\core\qkd_param.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\core\utils.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\solvers\__init__.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\solvers\fw2step.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\solvers\math_solver.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\modules\__init__.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\modules\description\__init__.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\modules\description\bb84_description.py
  criado: C:\Users\Mario\Documents\OpenQKD\openqkd_python\openqkd\modules\channel\__init__.py
  criado: C:\Users\Mario\Docume

In [1]:
import os, sys

base = r"C:\Users\Mario\Documents\openqkd_python"

print(f"Pasta existe: {os.path.exists(base)}")
print(f"\nConteúdo de openqkd_python:")
for item in os.listdir(base):
    print(f"  {item}")

print(f"\nsys.path[0] = {sys.path[0]}")


Pasta existe: False

Conteúdo de openqkd_python:


FileNotFoundError: [WinError 3] O sistema não pode encontrar o caminho especificado: 'C:\\Users\\Mario\\Documents\\openqkd_python'

In [2]:
import os, sys

base = r"C:\Users\Mario\Documents\OpenQKD\openqkd_python"

print(f"Pasta existe: {os.path.exists(base)}")
print(f"\nConteúdo de openqkd_python:")
for item in os.listdir(base):
    print(f"  {item}")

print(f"\nsys.path[0] = {sys.path[0]}")


Pasta existe: True

Conteúdo de openqkd_python:
  .ipynb_checkpoints
  .python-version
  .venv
  openqkd
  tests
  Untitled.ipynb

sys.path[0] = C:\Users\mario\.pyenv\pyenv-win\versions\3.11.9\python311.zip
